In [44]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F



In [20]:
# Pseudo code
vocab_size_4 = 4
id2word = ['<s>', 'a', 'b', 'c']
word2id = {
    '<s>': 0,
    'a': 1,
    'b': 2,
    'c': 3
}



In [21]:
print(id2word[0])
print(word2id['<s>'])

<s>
0


In [60]:
embed_dim_64 = 64

In [64]:
data = "aabbccaabbcc"

# Tokenize data + prepand a <s> token
input = torch.tensor([0,1,1,2,2,3,3,1,1,2,2,3,3])
actual = torch.tensor([1,1,2,2,3,3,1,1,2,2,3,3,1])
# actual_probs = torch.tensor([
#    # s  a  b  c
#    #[1, 0, 0, 0 ] <- this is shifted out of view in prediction
#     [0, 1, 0, 0 ],
#     [0, 1, 0, 0 ],
#     [0, 0, 1, 0 ],
#     [0, 0, 1, 0 ],
#     [0, 0, 0, 1 ],
#     [0, 0, 0, 1 ], 
#     [0, 1, 0, 0 ],
#     [0, 1, 0, 0 ],
#     [0, 0, 1, 0 ],
#     [0, 0, 1, 0 ],
#     [0, 0, 0, 1 ],
#     [0, 0, 0, 1 ], 
#     [0, 1, 0, 0 ]  # add the final predicted token — back to 'a'
# ]).float() # l = 13


In [23]:

embeddings_matrix = nn.Embedding(vocab_size_4, embed_dim_64)

W_Q = nn.Linear(embed_dim_64, embed_dim_64) # (linear layers)
W_K = nn.Linear(embed_dim_64, embed_dim_64)
W_V = nn.Linear(embed_dim_64, embed_dim_64)


In [24]:

seq_vectors = embeddings_matrix(input) # (13, 64)
seq_vectors.shape

torch.Size([13, 64])

In [27]:
q = W_Q(seq_vectors)
k = W_K(seq_vectors)
v = W_V(seq_vectors)
print(q.shape, k.shape, v.shape) # (13, 64) each

torch.Size([13, 64]) torch.Size([13, 64]) torch.Size([13, 64])


In [41]:
atn = q @ k.T
atn = atn / math.sqrt(embed_dim_64) # Not in Bes's code
atn.shape # (13,13)
print(atn)

tensor([[ 1.0156,  0.1075,  0.1075, -0.7594, -0.7594, -0.1806, -0.1806,  0.1075,
          0.1075, -0.7594, -0.7594, -0.1806, -0.1806],
        [-0.3585, -0.0232, -0.0232,  0.5904,  0.5904, -0.1323, -0.1323, -0.0232,
         -0.0232,  0.5904,  0.5904, -0.1323, -0.1323],
        [-0.3585, -0.0232, -0.0232,  0.5904,  0.5904, -0.1323, -0.1323, -0.0232,
         -0.0232,  0.5904,  0.5904, -0.1323, -0.1323],
        [-0.7417, -0.0420, -0.0420,  0.7171,  0.7171, -0.2562, -0.2562, -0.0420,
         -0.0420,  0.7171,  0.7171, -0.2562, -0.2562],
        [-0.7417, -0.0420, -0.0420,  0.7171,  0.7171, -0.2562, -0.2562, -0.0420,
         -0.0420,  0.7171,  0.7171, -0.2562, -0.2562],
        [ 0.3340,  0.1355,  0.1355,  0.2226,  0.2226,  0.0394,  0.0394,  0.1355,
          0.1355,  0.2226,  0.2226,  0.0394,  0.0394],
        [ 0.3340,  0.1355,  0.1355,  0.2226,  0.2226,  0.0394,  0.0394,  0.1355,
          0.1355,  0.2226,  0.2226,  0.0394,  0.0394],
        [-0.3585, -0.0232, -0.0232,  0.5904,  0.

In [42]:
neginf = torch.full_like(atn, float('-inf'))
mask = torch.triu(neginf, diagonal=1)

In [43]:
atn_masked = atn + mask
print(atn_masked)

tensor([[ 1.0156,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,
            -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.3585, -0.0232,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,
            -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.3585, -0.0232, -0.0232,    -inf,    -inf,    -inf,    -inf,    -inf,
            -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.7417, -0.0420, -0.0420,  0.7171,    -inf,    -inf,    -inf,    -inf,
            -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.7417, -0.0420, -0.0420,  0.7171,  0.7171,    -inf,    -inf,    -inf,
            -inf,    -inf,    -inf,    -inf,    -inf],
        [ 0.3340,  0.1355,  0.1355,  0.2226,  0.2226,  0.0394,    -inf,    -inf,
            -inf,    -inf,    -inf,    -inf,    -inf],
        [ 0.3340,  0.1355,  0.1355,  0.2226,  0.2226,  0.0394,  0.0394,    -inf,
            -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.3585, -0.0232, -0.0232,  0.5904,  0.

In [46]:
atn_probs = F.softmax(atn_masked, dim=-1) # softmax along rows
print(atn_probs)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000],
        [0.4169, 0.5831, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000],
        [0.2634, 0.3683, 0.3683, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000],
        [0.1072, 0.2158, 0.2158, 0.4611, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000],
        [0.0734, 0.1477, 0.1477, 0.3156, 0.3156, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000],
        [0.1933, 0.1585, 0.1585, 0.1729, 0.1729, 0.1440, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000],
        [0.1690, 0.1385, 0.1385, 0.1511, 0.1511, 0.1258, 0.1258, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000],
        [0.0777, 0.1087, 0.1087, 0.2007, 0.2007, 0.0974, 0.0974, 0.1087, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000],
        [0.0701,

In [49]:
out = atn_probs @ v
out.shape

torch.Size([13, 64])

In [47]:
ff = nn.Sequential(
    nn.Linear(embed_dim_64, embed_dim_64 * 4),
    nn.ReLU(),
    nn.Linear(embed_dim_64 * 4, embed_dim_64)
)

In [53]:
hidden = ff(out)
hidden.shape

torch.Size([13, 64])

In [54]:
proj = nn.Linear(embed_dim_64, vocab_size_4)

In [56]:
logits = proj(hidden)
logits.shape #(13, 4)

torch.Size([13, 4])

In [58]:
predicted = F.softmax(logits, dim=-1)
predicted.shape

torch.Size([13, 4])

In [67]:
loss = F.cross_entropy(predicted, actual)

In [68]:
optim = torch.optim.Adam()

tensor(1.3993, grad_fn=<NllLossBackward0>)